# customers.csv

In [82]:
import duckdb
import pandas as pd


con = duckdb.connect()
path = "../../source_database/seed_data/customers.csv"   

In [66]:
# 1. SCHEMA AUDIT — columns and inferred types
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ legacy_customer_ref │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ first_name          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ last_name           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ email               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ phone               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_of_birth       │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ postcode            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ loyalty_member_id   │ VARCHAR     │ YES     │ NULL    │ NULL  

In [83]:
# ── Completeness Audit: how many NULLs does each column have? ──

# Step 1: Get the list of column names automatically from the file.
# This way the same audit code works on ANY csv
columns_df = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
column_names = columns_df['column_name'].tolist()

# Step 2: Get the total row count once, so we can turn null counts into %.
total_rows = con.sql(f"SELECT COUNT(*) AS cnt FROM read_csv_auto('{path}')").df()['cnt'][0]
print(f"Total rows in file: {total_rows}")

# Step 3: For each column, ask DuckDB:
#   COUNT(*)        -> counts every row (NULL or not)
#   COUNT(col_name) -> counts only NON-null rows for that column
#   so COUNT(*) - COUNT(col_name) = how many rows are NULL in that column
null_report = []
for col in column_names:
    result = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) - COUNT({col}) AS null_count,
            ROUND(100.0 * (COUNT(*) - COUNT({col})) / COUNT(*), 2) AS null_percentage
        FROM read_csv_auto('{path}')
    """).df()
    null_report.append(result)

# Step 4: Stack all the per-column results into one table and show it.
import pandas as pd
null_report_df = pd.concat(null_report, ignore_index=True)
null_report_df

Total rows in file: 500


,column_name,null_count,null_percentage
0,customer_id,0,0.0
1,legacy_customer_ref,420,84.0
2,first_name,0,0.0
3,last_name,0,0.0
4,email,30,6.0
5,phone,0,0.0
6,date_of_birth,0,0.0
7,postcode,0,0.0
8,loyalty_member_id,121,24.2
9,acquisition_channel,0,0.0


In [6]:
# 3. UNIQUENESS AUDIT
# customer_id is the primary key — must be 100% unique.
# email, phone, legacy_customer_ref, loyalty_member_id SHOULD also be unique per person —
# a duplicate here usually means the same human was registered twice (a real business problem).
for col in ["customer_id", "email", "phone", "legacy_customer_ref", "loyalty_member_id"]:
    dupes = con.sql(f"""
        SELECT {col}, COUNT(*) AS row_count
        FROM read_csv_auto('{path}')
        GROUP BY {col}
        HAVING COUNT(*) > 1
        ORDER BY row_count DESC
    """).df()
    print(f"{col}: {len(dupes)} duplicated values")

customer_id: 0 duplicated values
email: 1 duplicated values
phone: 0 duplicated values
legacy_customer_ref: 1 duplicated values
loyalty_member_id: 1 duplicated values


In [8]:
# 4. VALIDITY / BUSINESS RULE AUDIT
# Checks that need real logic, not just "is it null"

# email must look like an email
bad_email = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE NOT REGEXP_MATCHES(email, '^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$')
""").fetchone()[0]
print(f"Malformed email addresses: {bad_email}")

# UK mobile numbers should be 10 digits starting with 7 (after removing the country/leading 0)
bad_phone = con.sql(f"""
    SELECT COUNT(*)
    FROM read_csv_auto('{path}', types={{'phone': 'VARCHAR'}})
    WHERE NOT REGEXP_MATCHES(phone, '^(0|\\+44)7[0-9]{{9}}$')
""").fetchone()[0]
print(f"Phone numbers not matching UK mobile format (07... or +447...): {bad_phone}")

# UK postcode format check (loose pattern — catches obvious junk, not 100% precise)
bad_postcode = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE NOT REGEXP_MATCHES(postcode, '^[A-Z]{{1,2}}[0-9][A-Z0-9]? [0-9][A-Z]{{2}}$')
""").fetchone()[0]
print(f"Postcodes not matching UK format: {bad_postcode}")

# date_of_birth should produce a sane age (say, 16-100) — catches typos like 1899 or 2020
bad_age = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE DATE_DIFF('year', CAST(date_of_birth AS DATE), CURRENT_DATE) NOT BETWEEN 16 AND 100
""").fetchone()[0]
print(f"Customers with implausible age (DOB outside 16-100 yrs): {bad_age}")

# gdpr_consent_date can't be in the future
future_consent = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE CAST(gdpr_consent_date AS DATE) > CURRENT_DATE
""").fetchone()[0]
print(f"gdpr_consent_date in the future: {future_consent}")

# created_at should not be after updated_at
bad_update_order = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE CAST(updated_at AS TIMESTAMP) < CAST(created_at AS TIMESTAMP)
""").fetchone()[0]
print(f"updated_at earlier than created_at: {bad_update_order}")

Malformed email addresses: 0
Phone numbers not matching UK mobile format (07... or +447...): 0
Postcodes not matching UK format: 0
Customers with implausible age (DOB outside 16-100 yrs): 0
gdpr_consent_date in the future: 0
updated_at earlier than created_at: 0


In [9]:
# 5. DISTRIBUTION / OUTLIER AUDIT — age distribution
con.sql(f"""
    SELECT
        MIN(DATE_DIFF('year', CAST(date_of_birth AS DATE), CURRENT_DATE)) AS min_age,
        MAX(DATE_DIFF('year', CAST(date_of_birth AS DATE), CURRENT_DATE)) AS max_age,
        AVG(DATE_DIFF('year', CAST(date_of_birth AS DATE), CURRENT_DATE)) AS avg_age
    FROM read_csv_auto('{path}')
""").show()

┌─────────┬─────────┬─────────┐
│ min_age │ max_age │ avg_age │
│  int64  │  int64  │ double  │
├─────────┼─────────┼─────────┤
│      26 │      71 │   47.56 │
└─────────┴─────────┴─────────┘



In [10]:
# 6. FRESHNESS / VOLUME AUDIT
con.sql(f"""
    SELECT
        MIN(CAST(created_at AS TIMESTAMP)) AS earliest_signup,
        MAX(CAST(created_at AS TIMESTAMP)) AS latest_signup,
        COUNT(*) AS total_rows
    FROM read_csv_auto('{path}')
""").show()

# signups per year — spot any odd concentration
con.sql(f"""
    SELECT DATE_TRUNC('year', CAST(created_at AS TIMESTAMP)) AS year, COUNT(*) AS signups
    FROM read_csv_auto('{path}')
    GROUP BY year ORDER BY year
""").show()

┌─────────────────────┬─────────────────────┬────────────┐
│   earliest_signup   │    latest_signup    │ total_rows │
│      timestamp      │      timestamp      │   int64    │
├─────────────────────┼─────────────────────┼────────────┤
│ 2015-01-01 08:50:00 │ 2024-12-31 10:58:00 │        500 │
└─────────────────────┴─────────────────────┴────────────┘

┌────────────┬─────────┐
│    year    │ signups │
│    date    │  int64  │
├────────────┼─────────┤
│ 2015-01-01 │      35 │
│ 2016-01-01 │      60 │
│ 2017-01-01 │      41 │
│ 2018-01-01 │      55 │
│ 2019-01-01 │      42 │
│ 2020-01-01 │      45 │
│ 2021-01-01 │      54 │
│ 2022-01-01 │      61 │
│ 2023-01-01 │      49 │
│ 2024-01-01 │      58 │
├────────────┴─────────┤
│ 10 rows    2 columns │
└──────────────────────┘



In [11]:
# 7. FORMAT CONSISTENCY AUDIT
con.sql(f"SELECT DISTINCT acquisition_channel FROM read_csv_auto('{path}')").show()

# gdpr_consent should logically be TRUE/FALSE only — sanity check
con.sql(f"SELECT DISTINCT gdpr_consent FROM read_csv_auto('{path}')").show()
con.sql(f"SELECT DISTINCT is_active FROM read_csv_auto('{path}')").show()

# email casing — inconsistent case in real systems is common and causes false "different" values downstream
mixed_case_emails = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE email != LOWER(email)
""").fetchone()[0]
print(f"Emails with uppercase characters: {mixed_case_emails}")

┌─────────────────────┐
│ acquisition_channel │
│       varchar       │
├─────────────────────┤
│ click_collect       │
│ online              │
│ store               │
│ freshmart_migration │
└─────────────────────┘

┌──────────────┐
│ gdpr_consent │
│   boolean    │
├──────────────┤
│ false        │
│ true         │
└──────────────┘

┌───────────┐
│ is_active │
│  boolean  │
├───────────┤
│ false     │
│ true      │
└───────────┘

Emails with uppercase characters: 0


In [12]:
# 8. GDPR / PII AWARENESS CHECK — specific to customer data, not covered in orders
# You don't need to fix this now, but flag it: which columns are PII and need masking in Silver
# per your ADR-001 (SHA-256 masking requirement)
pii_columns = ["email", "phone", "date_of_birth", "postcode", "first_name", "last_name"]
print("PII columns requiring masking at Silver layer:", pii_columns)

consent_false_but_used = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE gdpr_consent = FALSE
""").fetchone()[0]
print(f"Customers with gdpr_consent = FALSE: {consent_false_but_used} "
      f"— these should be excluded from marketing-related Gold models")

PII columns requiring masking at Silver layer: ['email', 'phone', 'date_of_birth', 'postcode', 'first_name', 'last_name']
Customers with gdpr_consent = FALSE: 34 — these should be excluded from marketing-related Gold models


In [13]:
non_consenting = con.sql(f"""
    SELECT customer_id, first_name, last_name, email, gdpr_consent, gdpr_consent_date
    FROM read_csv_auto('{path}')
    WHERE gdpr_consent = FALSE
    ORDER BY customer_id
""").df()

print(f"Total non-consenting customers: {len(non_consenting)}")
print(f"customer_id range: {non_consenting['customer_id'].min()} to {non_consenting['customer_id'].max()}")
non_consenting

Total non-consenting customers: 34
customer_id range: 2 to 469


,customer_id,first_name,last_name,email,gdpr_consent,gdpr_consent_date
0,2,Daniel,Garcia,customer2@yahoo.co.uk,False,2024-08-04
1,14,Sophie,Wilson,customer14@yahoo.co.uk,False,2024-09-21
2,58,Oliver,Garcia,customer58@outlook.co.uk,False,2022-03-15
3,107,Jack,MacDonald,customer107@yahoo.co.uk,False,2025-04-16
4,111,Lily,O Brien,customer111@gmail.co.uk,False,2025-03-06
5,148,George,Evans,customer148@outlook.com,False,2018-07-13
6,153,Charlotte,Johnson,customer153@gmail.co.uk,False,NaT
7,156,Edward,MacDonald,customer156@gmail.com,False,2021-11-04
8,164,Lily,Murphy,customer164@yahoo.com,False,2024-05-01
9,171,Edward,Sharma,customer171@gmail.co.uk,False,2019-07-27


In [51]:
# 9. Distribution anomaly check
for col in ['acquisition_channel', 'gdpr_consent', 'is_active']:
    print(f"\n-- {col} --")
    print(con.sql(f"""
        SELECT {col}, COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 1) AS pct
        FROM read_csv_auto('{path}') GROUP BY {col} ORDER BY n DESC
    """).df())

print("\n-- email domain --")
print(con.sql(f"""
    SELECT split_part(email, '@', 2) AS domain, COUNT(*) AS n,
           ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 1) AS pct
    FROM read_csv_auto('{path}') GROUP BY domain ORDER BY n DESC
""").df())


-- acquisition_channel --
   acquisition_channel    n   pct
0                store  178  35.6
1  freshmart_migration  140  28.0
2               online  125  25.0
3        click_collect   57  11.4

-- gdpr_consent --
   gdpr_consent    n   pct
0          True  466  93.2
1         False   34   6.8

-- is_active --
   is_active    n   pct
0       True  478  95.6
1      False   22   4.4

-- email domain --
          domain    n   pct
0    gmail.co.uk  120  24.0
1    yahoo.co.uk  118  23.6
2  outlook.co.uk  115  23.0
3      gmail.com   40   8.0
4      yahoo.com   39   7.8
5    outlook.com   38   7.6
6                  30   6.0


In [81]:
# --- 10. Timezone drift: hour-of-day distribution ---
print(con.sql(f"""
    SELECT EXTRACT(hour FROM created_at) AS hour, COUNT(*) AS n
    FROM read_csv_auto('{path}') GROUP BY hour ORDER BY hour
""").df())

# --- 2. Is this timestamp column even real, or a placeholder? ---
# A genuine "last updated" column should have many distinct values spread
# over time. If it collapses to ONE value for every row, it's not tracking
# anything — worth knowing before you build change-tracking logic on it.
print(con.sql(f"""
    SELECT updated_at, COUNT(*) AS n
    FROM read_csv_auto('{path}') GROUP BY updated_at ORDER BY n DESC
""").df())

# --- 3. Sequential gaps in the ID column ---
print(con.sql(f"""
    WITH bounds AS (SELECT MIN(customer_id) mn, MAX(customer_id) mx FROM read_csv_auto('{path}')),
    spine AS (SELECT UNNEST(generate_series(mn, mx)) AS id FROM bounds)
    SELECT s.id AS missing_customer_id
    FROM spine s LEFT JOIN read_csv_auto('{path}') c ON s.id = c.customer_id
    WHERE c.customer_id IS NULL
""").df())

# --- 4. Logical time order: nothing should be "updated" before it was "created" ---
print(con.sql(f"""
    SELECT COUNT(*) AS backwards_count
    FROM read_csv_auto('{path}') WHERE updated_at < created_at
""").df())

   backwards_count
0                0


In [69]:
results = []

# 1. Hour-of-day spread (timezone drift check)
hours = con.sql(f"SELECT EXTRACT(hour FROM created_at) AS hour, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY hour").df()
results.append({
    'check': 'created_at hour-of-day coverage',
    'result': f"{hours['hour'].nunique()} distinct hours used (range {int(hours['hour'].min())}-{int(hours['hour'].max())})",
    'status': 'OK'
})

# 2. Is updated_at a real timestamp or a constant placeholder?
uniq_updated = con.sql(f"SELECT COUNT(DISTINCT updated_at) AS n FROM read_csv_auto('{path}')").fetchone()[0]
results.append({
    'check': 'updated_at distinct values',
    'result': f"{uniq_updated} distinct value(s) across 500 rows",
    'status': 'OK' if uniq_updated > 1 else '⚠️ FLAG — placeholder, not real'
})

# 3. Gaps in customer_id sequence
gaps = con.sql(f"""
    WITH bounds AS (SELECT MIN(customer_id) mn, MAX(customer_id) mx FROM read_csv_auto('{path}')),
    spine AS (SELECT UNNEST(generate_series(mn, mx)) AS id FROM bounds)
    SELECT COUNT(*) AS n FROM spine s LEFT JOIN read_csv_auto('{path}') c ON s.id = c.customer_id WHERE c.customer_id IS NULL
""").fetchone()[0]
results.append({
    'check': 'customer_id sequence gaps',
    'result': f"{gaps} missing ID(s)",
    'status': 'OK' if gaps == 0 else '⚠️ FLAG'
})

# 4. Logical order: updated_at should never be before created_at
backwards = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}') WHERE updated_at < created_at").fetchone()[0]
results.append({
    'check': 'updated_at before created_at',
    'result': f"{backwards} row(s)",
    'status': 'OK' if backwards == 0 else '⚠️ FLAG'
})

summary = pd.DataFrame(results)
summary

,check,result,status
0,created_at hour-of-day coverage,17 distinct hours used (range 6-22),OK
1,updated_at distinct values,1 distinct value(s) across 500 rows,"⚠️ FLAG — placeholder, not real"
2,customer_id sequence gaps,0 missing ID(s),OK
3,updated_at before created_at,0 row(s),OK


In [71]:
# 11. Format consistency: Flag columns that contain mixed data types
def format_consistency_check(con, path):
    # Step 1: get every column name in the file
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()

    all_rows = []
    for col in columns:
        # Step 2: for THIS column, classify every value into a bucket
        breakdown = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                CASE
                    WHEN "{col}" IS NULL OR TRIM("{col}") = '' THEN 'NULL/EMPTY'
                    WHEN TRY_CAST("{col}" AS BIGINT) IS NOT NULL THEN 'INTEGER'
                    WHEN TRY_CAST("{col}" AS DOUBLE) IS NOT NULL THEN 'DECIMAL'
                    WHEN TRY_CAST("{col}" AS DATE) IS NOT NULL THEN 'DATE'
                    WHEN TRY_CAST("{col}" AS TIMESTAMP) IS NOT NULL THEN 'TIMESTAMP'
                    WHEN lower("{col}") IN ('true','false') THEN 'BOOLEAN'
                    ELSE 'TEXT'
                END AS detected_type,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            GROUP BY detected_type
            ORDER BY row_count DESC
        """).df()
        all_rows.append(breakdown)

    result = pd.concat(all_rows, ignore_index=True)

    # Step 3: flag a column only if it has 2+ real (non-null) types
    n_types = result[result['detected_type'] != 'NULL/EMPTY'].groupby('column_name')['detected_type'].transform('nunique')
    result['is_mixed'] = (n_types.reindex(result.index).fillna(0) > 1)
    return result

report = format_consistency_check(con, path)
report                       # full breakdown — bare, no print(), so it renders as a table

,column_name,detected_type,row_count,is_mixed
0,customer_id,INTEGER,500,False
1,legacy_customer_ref,NULL/EMPTY,420,False
2,legacy_customer_ref,TEXT,80,False
3,first_name,TEXT,500,False
4,last_name,TEXT,500,False
5,email,TEXT,470,False
6,email,NULL/EMPTY,30,False
7,phone,INTEGER,500,False
8,date_of_birth,DATE,500,False
9,postcode,TEXT,500,False


In [76]:
# 12. Sentinel/Default value usage:

# Part 1: scan for known/common sentinel patterns
SENTINEL_VALUES = [
    '-1', '0', '999', '9999', '99999', '-999',
    'NA', 'N/A', 'NULL', 'NONE', 'UNKNOWN', 'UNK', 'TBD', '-', '?',
    '1900-01-01', '1970-01-01', '9999-12-31', '0001-01-01'
]

def sentinel_scan(con, path, sentinel_values):
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()
    values_sql = ", ".join(f"'{v.upper()}'" for v in sentinel_values)

    all_rows = []
    for col in columns:
        df = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                "{col}" AS value,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            WHERE upper(TRIM("{col}")) IN ({values_sql})
            GROUP BY "{col}"
            ORDER BY row_count DESC
        """).df()
        all_rows.append(df)

    return pd.concat(all_rows, ignore_index=True)

sentinel_report = sentinel_scan(con, path, SENTINEL_VALUES)
sentinel_report


,column_name,value,row_count
0,date_of_birth,1970-01-01,1


In [78]:
# Part 2: catch sentinels NOT on my list

def dominant_value_scan(con, path, threshold_pct=15, min_distinct_to_check=10):
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()

    all_rows = []
    for _, row in schema.iterrows():
        col, dtype = row['column_name'], row['column_type']

        # BOOLEAN can only ever have 2 values — dominance there is
        # guaranteed, never a sentinel signal. Skip entirely.
        if dtype == 'BOOLEAN':
            continue

        stats = con.sql(f"""
            SELECT COUNT(DISTINCT "{col}") AS distinct_count, COUNT("{col}") AS non_null_count
            FROM read_csv_auto('{path}', all_varchar=True)
        """).df().iloc[0]

        # A small fixed set of real categories (like acquisition_channel)
        # is expected to have a "leading" value — skip those too, UNLESS
        # it's a date/timestamp, which should almost never repeat at all.
        is_time_col = dtype in ('DATE', 'TIMESTAMP')
        if stats['distinct_count'] < min_distinct_to_check and not is_time_col:
            continue

        top = con.sql(f"""
            SELECT "{col}" AS top_value, COUNT(*) AS n
            FROM read_csv_auto('{path}', all_varchar=True)
            WHERE "{col}" IS NOT NULL
            GROUP BY "{col}" ORDER BY n DESC LIMIT 1
        """).df().iloc[0]

        pct = round(100.0 * top['n'] / stats['non_null_count'], 1)
        # dates/timestamps get a much stricter bar — a real one should
        # basically never repeat, so even modest concentration is suspicious
        effective_threshold = 5 if is_time_col else threshold_pct

        all_rows.append({
            'column_name': col,
            'column_type': dtype,
            'distinct_count': int(stats['distinct_count']),
            'top_value': top['top_value'],
            'n': int(top['n']),
            'pct_of_column': pct,
            'flag': pct >= effective_threshold
        })

    return pd.DataFrame(all_rows).sort_values('pct_of_column', ascending=False)

dominant_report = dominant_value_scan(con, path)
dominant_report

,column_name,column_type,distinct_count,top_value,n,pct_of_column,flag
11,updated_at,TIMESTAMP,1,2026-01-01 00:00:00,500,100.0,True
3,last_name,VARCHAR,20,Brown,35,7.0,False
2,first_name,VARCHAR,30,Lily,27,5.4,False
1,legacy_customer_ref,VARCHAR,80,LCR000003,1,1.2,False
9,gdpr_consent_date,DATE,422,2018-09-11,3,0.7,False
6,date_of_birth,DATE,490,1986-09-06,2,0.4,False
8,loyalty_member_id,VARCHAR,379,NR0000020,1,0.3,False
0,customer_id,BIGINT,500,1,1,0.2,False
4,email,VARCHAR,470,customer14@yahoo.co.uk,1,0.2,False
5,phone,VARCHAR,500,07736855845,1,0.2,False


# customers.xlsx

In [29]:
# ==========================================================
# CUSTOMERS.XLSX AUDIT — same checks as customers.csv
# ==========================================================
import duckdb
import pandas as pd

xlsx_path = "../../source_database/seed_data/customers.xlsx"
customers_xlsx = pd.read_excel(xlsx_path, dtype={'phone': str})

# side-step DuckDB's register()/read_xlsx issues — export to CSV, then read that
customers_xlsx.to_csv("customers_xlsx_export.csv", index=False)

con = duckdb.connect()
path = "customers_xlsx_export.csv"   # everything below uses this variable name

In [30]:
# 1. SCHEMA AUDIT — columns and inferred types
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ legacy_customer_ref │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ first_name          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ last_name           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ email               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ phone               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_of_birth       │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ postcode            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ loyalty_member_id   │ VARCHAR     │ YES     │ NULL    │ NULL  

In [31]:
# 2. COMPLETENESS AUDIT — null count and % per column
total_rows = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
print(f"Total rows: {total_rows}")

cols = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()['column_name'].tolist()
for col in cols:
    null_count = con.sql(f"SELECT COUNT(*) - COUNT({col}) FROM read_csv_auto('{path}')").fetchone()[0]
    pct = round(null_count / total_rows * 100, 2)
    print(f"{col}: {null_count} nulls ({pct}%)")

Total rows: 500
customer_id: 0 nulls (0.0%)
legacy_customer_ref: 350 nulls (70.0%)
first_name: 0 nulls (0.0%)
last_name: 0 nulls (0.0%)
email: 22 nulls (4.4%)
phone: 0 nulls (0.0%)
date_of_birth: 0 nulls (0.0%)
postcode: 0 nulls (0.0%)
loyalty_member_id: 0 nulls (0.0%)
acquisition_channel: 0 nulls (0.0%)
gdpr_consent: 0 nulls (0.0%)
gdpr_consent_date: 0 nulls (0.0%)
is_active: 0 nulls (0.0%)
created_at: 0 nulls (0.0%)
updated_at: 0 nulls (0.0%)


In [32]:
# 3. UNIQUENESS AUDIT
# customer_id is the primary key — must be 100% unique.
# email, phone, legacy_customer_ref, loyalty_member_id should also be unique per person.
for col in ["customer_id", "email", "phone", "legacy_customer_ref", "loyalty_member_id"]:
    dupes = con.sql(f"""
        SELECT {col}, COUNT(*) AS row_count
        FROM read_csv_auto('{path}')
        GROUP BY {col}
        HAVING COUNT(*) > 1
        ORDER BY row_count DESC
    """).df()
    print(f"{col}: {len(dupes)} duplicated values")

customer_id: 0 duplicated values
email: 1 duplicated values
phone: 0 duplicated values
legacy_customer_ref: 1 duplicated values
loyalty_member_id: 0 duplicated values


In [33]:
# 4. VALIDITY / BUSINESS RULE AUDIT

# email must look like an email
bad_email = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE NOT REGEXP_MATCHES(email, '^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$')
""").fetchone()[0]
print(f"Malformed email addresses: {bad_email}")

# phone — force VARCHAR so a leading 0 isn't silently stripped by type inference,
# accept both UK domestic (0...) and international (+44...) formats
bad_phone = con.sql(f"""
    SELECT COUNT(*)
    FROM read_csv_auto('{path}', types={{'phone': 'VARCHAR'}})
    WHERE NOT REGEXP_MATCHES(phone, '^(0|\\+44)7[0-9]{{9}}$')
""").fetchone()[0]
print(f"Phone numbers not matching UK mobile format (07... or +447...): {bad_phone}")

# UK postcode format check (loose pattern)
bad_postcode = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE NOT REGEXP_MATCHES(postcode, '^[A-Z]{{1,2}}[0-9][A-Z0-9]? [0-9][A-Z]{{2}}$')
""").fetchone()[0]
print(f"Postcodes not matching UK format: {bad_postcode}")

# date_of_birth should give a sane age (16-100)
bad_age = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE DATE_DIFF('year', CAST(date_of_birth AS DATE), CURRENT_DATE) NOT BETWEEN 16 AND 100
""").fetchone()[0]
print(f"Customers with implausible age (DOB outside 16-100 yrs): {bad_age}")

# gdpr_consent_date can't be in the future
future_consent = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE CAST(gdpr_consent_date AS DATE) > CURRENT_DATE
""").fetchone()[0]
print(f"gdpr_consent_date in the future: {future_consent}")

# created_at should not be after updated_at
bad_update_order = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE CAST(updated_at AS TIMESTAMP) < CAST(created_at AS TIMESTAMP)
""").fetchone()[0]
print(f"updated_at earlier than created_at: {bad_update_order}")

Malformed email addresses: 0
Phone numbers not matching UK mobile format (07... or +447...): 0
Postcodes not matching UK format: 0
Customers with implausible age (DOB outside 16-100 yrs): 0
gdpr_consent_date in the future: 0
updated_at earlier than created_at: 0


In [34]:
# 5. DISTRIBUTION / OUTLIER AUDIT — age distribution
con.sql(f"""
    SELECT
        MIN(DATE_DIFF('year', CAST(date_of_birth AS DATE), CURRENT_DATE)) AS min_age,
        MAX(DATE_DIFF('year', CAST(date_of_birth AS DATE), CURRENT_DATE)) AS max_age,
        AVG(DATE_DIFF('year', CAST(date_of_birth AS DATE), CURRENT_DATE)) AS avg_age
    FROM read_csv_auto('{path}')
""").show()

┌─────────┬─────────┬─────────┐
│ min_age │ max_age │ avg_age │
│  int64  │  int64  │ double  │
├─────────┼─────────┼─────────┤
│      26 │      71 │  48.288 │
└─────────┴─────────┴─────────┘



In [35]:
# 6. FRESHNESS / VOLUME AUDIT
con.sql(f"""
    SELECT
        MIN(CAST(created_at AS TIMESTAMP)) AS earliest_signup,
        MAX(CAST(created_at AS TIMESTAMP)) AS latest_signup,
        COUNT(*) AS total_rows
    FROM read_csv_auto('{path}')
""").show()

con.sql(f"""
    SELECT DATE_TRUNC('year', CAST(created_at AS TIMESTAMP)) AS year, COUNT(*) AS signups
    FROM read_csv_auto('{path}')
    GROUP BY year ORDER BY year
""").show()

┌─────────────────────┬─────────────────────┬────────────┐
│   earliest_signup   │    latest_signup    │ total_rows │
│      timestamp      │      timestamp      │   int64    │
├─────────────────────┼─────────────────────┼────────────┤
│ 2018-01-06 00:00:00 │ 2025-12-14 00:00:00 │        500 │
└─────────────────────┴─────────────────────┴────────────┘

┌────────────┬─────────┐
│    year    │ signups │
│    date    │  int64  │
├────────────┼─────────┤
│ 2018-01-01 │      69 │
│ 2019-01-01 │      71 │
│ 2020-01-01 │      55 │
│ 2021-01-01 │      73 │
│ 2022-01-01 │      56 │
│ 2023-01-01 │      57 │
│ 2024-01-01 │      56 │
│ 2025-01-01 │      63 │
└────────────┴─────────┘



In [36]:
# 7. FORMAT CONSISTENCY AUDIT
con.sql(f"SELECT DISTINCT acquisition_channel FROM read_csv_auto('{path}')").show()
con.sql(f"SELECT DISTINCT gdpr_consent FROM read_csv_auto('{path}')").show()
con.sql(f"SELECT DISTINCT is_active FROM read_csv_auto('{path}')").show()

mixed_case_emails = con.sql(f"""
    SELECT COUNT(*) FROM read_csv_auto('{path}')
    WHERE email != LOWER(email)
""").fetchone()[0]
print(f"Emails with uppercase characters: {mixed_case_emails}")

┌─────────────────────┐
│ acquisition_channel │
│       varchar       │
├─────────────────────┤
│ freshmart_migration │
│ store               │
│ referral            │
│ click_collect       │
│ online              │
└─────────────────────┘

┌──────────────┐
│ gdpr_consent │
│   boolean    │
├──────────────┤
│ true         │
│ false        │
└──────────────┘

┌───────────┐
│ is_active │
│  boolean  │
├───────────┤
│ false     │
│ true      │
└───────────┘

Emails with uppercase characters: 0


In [38]:
# 8. GDPR / PII AWARENESS CHECK
pii_columns = ["email", "phone", "date_of_birth", "postcode", "first_name", "last_name"]
print("PII columns requiring masking at Silver layer:", pii_columns)

non_consenting = con.sql(f"""
    SELECT customer_id, first_name, last_name, email, gdpr_consent, gdpr_consent_date
    FROM read_csv_auto('{path}')
    WHERE gdpr_consent = FALSE
    ORDER BY customer_id
""").df()
print(f"Customers with gdpr_consent = FALSE: {len(non_consenting)}")

PII columns requiring masking at Silver layer: ['email', 'phone', 'date_of_birth', 'postcode', 'first_name', 'last_name']
Customers with gdpr_consent = FALSE: 44


# comparison between csv and xlsx

In [39]:
# ==========================================================
# CUSTOMERS.CSV vs CUSTOMERS.XLSX — FULL COMPARISON AUDIT
# ==========================================================
import pandas as pd

csv_path = "../../source_database/seed_data/customers.csv"
xlsx_path = "../../source_database/seed_data/customers.xlsx"

# force phone to load as text in BOTH files — avoids the leading-zero bug
# from earlier corrupting this comparison
csv_df = pd.read_csv(csv_path, dtype={'phone': str})
xlsx_df = pd.read_excel(xlsx_path, dtype={'phone': str})

print(f"customers.csv  -> {csv_df.shape[0]} rows, {csv_df.shape[1]} columns")
print(f"customers.xlsx -> {xlsx_df.shape[0]} rows, {xlsx_df.shape[1]} columns")

customers.csv  -> 500 rows, 15 columns
customers.xlsx -> 500 rows, 15 columns


In [40]:
# STEP 1 — SCHEMA COMPARISON
csv_cols = set(csv_df.columns)
xlsx_cols = set(xlsx_df.columns)

only_in_csv  = csv_cols - xlsx_cols
only_in_xlsx = xlsx_cols - csv_cols
shared_cols  = csv_cols & xlsx_cols

print("Columns only in customers.csv :", sorted(only_in_csv))
print("Columns only in customers.xlsx:", sorted(only_in_xlsx))
print("Columns in both files         :", sorted(shared_cols))

Columns only in customers.csv : []
Columns only in customers.xlsx: []
Columns in both files         : ['acquisition_channel', 'created_at', 'customer_id', 'date_of_birth', 'email', 'first_name', 'gdpr_consent', 'gdpr_consent_date', 'is_active', 'last_name', 'legacy_customer_ref', 'loyalty_member_id', 'phone', 'postcode', 'updated_at']


In [41]:
# STEP 2 — DATA TYPE COMPARISON
dtype_compare = pd.DataFrame({
    "csv_dtype":  csv_df[sorted(shared_cols)].dtypes,
    "xlsx_dtype": xlsx_df[sorted(shared_cols)].dtypes
})
dtype_compare["same_dtype"] = dtype_compare["csv_dtype"] == dtype_compare["xlsx_dtype"]
dtype_compare

,csv_dtype,xlsx_dtype,same_dtype
acquisition_channel,str,str,True
created_at,str,str,True
customer_id,int64,int64,True
date_of_birth,str,str,True
email,str,str,True
first_name,str,str,True
gdpr_consent,bool,bool,True
gdpr_consent_date,str,str,True
is_active,bool,bool,True
last_name,str,str,True


In [42]:
# STEP 3 — customer_id OVERLAP
csv_ids  = set(csv_df["customer_id"])
xlsx_ids = set(xlsx_df["customer_id"])

only_in_csv_ids  = csv_ids - xlsx_ids
only_in_xlsx_ids = xlsx_ids - csv_ids
shared_ids       = csv_ids & xlsx_ids

print(f"customer_id only in csv : {len(only_in_csv_ids)}")
print(f"customer_id only in xlsx: {len(only_in_xlsx_ids)}")
print(f"customer_id in both     : {len(shared_ids)}")

if only_in_xlsx_ids:
    print("customer_ids only in xlsx (min-max):", min(only_in_xlsx_ids), "-", max(only_in_xlsx_ids))
if only_in_csv_ids:
    print("customer_ids only in csv (min-max):", min(only_in_csv_ids), "-", max(only_in_csv_ids))

customer_id only in csv : 0
customer_id only in xlsx: 0
customer_id in both     : 500


In [44]:
# STEP 5 — FIELD-BY-FIELD COMPARISON FOR customer_ids PRESENT IN BOTH FILES
# The real question: for the SAME customer_id, do the two files agree?
compare_cols = sorted(shared_cols - {"customer_id"})

merged = csv_df.merge(xlsx_df, on="customer_id", suffixes=("_csv", "_xlsx"))
print(f"Rows compared (customer_id present in both files): {len(merged)}")

mismatch_summary = {}
for col in compare_cols:
    mismatches = merged[f"{col}_csv"] != merged[f"{col}_xlsx"]
    mismatch_summary[col] = {
        "mismatches": int(mismatches.sum()),
        "pct": round(mismatches.sum() / len(merged) * 100, 2)
    }

mismatch_df = pd.DataFrame(mismatch_summary).T.sort_values("pct", ascending=False)
mismatch_df

Rows compared (customer_id present in both files): 500


,mismatches,pct
created_at,500.0,100.0
date_of_birth,500.0,100.0
gdpr_consent_date,500.0,100.0
phone,500.0,100.0
postcode,499.0,99.8
first_name,495.0,99.0
last_name,482.0,96.4
legacy_customer_ref,420.0,84.0
email,398.0,79.6
acquisition_channel,344.0,68.8


In [45]:
# STEP 6 — SAMPLE EVIDENCE ROWS
# Pick whichever column had the highest mismatch % in Step 5 and inspect real examples
worst_col = mismatch_df.index[0]
sample_mismatch = merged[merged[f"{worst_col}_csv"] != merged[f"{worst_col}_xlsx"]][
    ["customer_id", "first_name_csv", "first_name_xlsx",
     f"{worst_col}_csv", f"{worst_col}_xlsx"]
].head(10)
print(f"Sample mismatches on: {worst_col}")
sample_mismatch

Sample mismatches on: created_at


,customer_id,first_name_csv,first_name_xlsx,created_at_csv,created_at_xlsx
0,1,Grace,Mason,2023-08-25 10:40:00,2021-01-07 00:00:00
1,2,Daniel,Ethan,2016-06-06 07:19:00,2020-06-06 00:00:00
2,3,Charlotte,Amelia,2024-02-03 11:56:00,2018-07-10 00:00:00
3,4,Amelia,Sophie,2024-03-14 13:14:00,2021-12-20 00:00:00
4,5,James,Ava,2017-12-05 11:37:00,2024-07-19 00:00:00
5,6,Daniel,Samuel,2021-01-14 12:23:00,2025-03-22 00:00:00
6,7,Amelia,David,2019-04-02 11:39:00,2022-02-18 00:00:00
7,8,William,Ethan,2018-03-08 22:42:00,2019-11-15 00:00:00
8,9,Samuel,Henry,2022-01-23 12:45:00,2025-12-14 00:00:00
9,10,Joseph,Henry,2024-07-06 09:29:00,2023-02-01 00:00:00


In [46]:
# STEP 7 — FINAL SUMMARY (copy this block straight into your documentation)
print("=== customers.csv vs customers.xlsx — comparison summary ===")
print(f"customers.csv : {len(csv_df)} rows, {len(csv_df.columns)} columns")
print(f"customers.xlsx: {len(xlsx_df)} rows, {len(xlsx_df.columns)} columns")
print(f"Columns only in csv : {sorted(only_in_csv)}")
print(f"Columns only in xlsx: {sorted(only_in_xlsx)}")
print(f"customer_id shared between files: {len(shared_ids)}")
print(f"customer_id only in xlsx: {len(only_in_xlsx_ids)}")
print()
print("Field mismatch rate on shared customer_ids:")
print(mismatch_df)

=== customers.csv vs customers.xlsx — comparison summary ===
customers.csv : 500 rows, 15 columns
customers.xlsx: 500 rows, 15 columns
Columns only in csv : []
Columns only in xlsx: []
customer_id shared between files: 500
customer_id only in xlsx: 0

Field mismatch rate on shared customer_ids:
                     mismatches    pct
created_at                500.0  100.0
date_of_birth             500.0  100.0
gdpr_consent_date         500.0  100.0
phone                     500.0  100.0
postcode                  499.0   99.8
first_name                495.0   99.0
last_name                 482.0   96.4
legacy_customer_ref       420.0   84.0
email                     398.0   79.6
acquisition_channel       344.0   68.8
loyalty_member_id         121.0   24.2
gdpr_consent               72.0   14.4
is_active                  52.0   10.4
updated_at                  0.0    0.0
